# Phase 6 — the random-reward control (seed 3407)

**Runtime: A100.** ~1.5 h, ~15–20 units. One lineage is enough for the control.

**The question this answers:** was GRPO's gain caused by the EXECUTION signal,
or would any RL-shaped shaking of the model (KL anchor, format pressure,
sampling regime) produce it? (Spurious Rewards, arXiv 2506.10947.)

**Design:** byte-identical to the seed-3407 GRPO run in every respect except
ONE — the reward returned to the trainer is a **uniform random number**,
independent of the code. The real grader still runs silently in the background
so we can watch what happens to TRUE quality while the model chases noise
(logged to Drive, never returned to the trainer).

**Pre-committed predictions:**
- Random ≈ SFT baseline (no gain) → GRPO's +2.3/+3.3 on this lineage was real
- Random ≈ GRPO (same gain) → our 'signal' was spurious; report that honestly
- Random ≪ SFT (degradation) → noise actively hurts; also worth reporting

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc, uuid
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
PHASE5 = '/content/drive/MyDrive/rl-post-training/phase5'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts', 'reward'):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
train_bugs = [b for b in bugs if b['split'] == 'train']
rl_bugs = [b for b in train_bugs if routing.get(b['id']) == 'rl']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
dev_clean = [r for r in restraint if r['split'] == 'dev']
SEED = 3407
print(len(rl_bugs), 'RL-pile bugs |', len(dev_bugs), 'dev |', len(dev_clean), 'dev clean')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# Silent true-grader (monitoring only) + the RANDOM reward the trainer sees
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
SENTINEL = uuid.uuid4().hex

def run_tests(code, rec, timeout=10):
    tests = list(rec['test_list'])
    try:
        compile(code, '<cand>', 'exec')
    except Exception:
        return False
    harness = (
        '\n'.join(list(rec['test_imports'])) + '\n' + code + '\n'
        + '__zz_pass = 0\n'
        + f'for __zz_t in {tests!r}:\n'
        + '    try:\n        exec(__zz_t)\n        __zz_pass += 1\n'
        + '    except BaseException:\n        pass\n'
        + f'print("{SENTINEL}", __zz_pass)\n'
    )
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(harness); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, timeout=timeout)
        n = 0
        for line in r.stdout.decode(errors='ignore').splitlines():
            parts = line.split()
            if len(parts) == 2 and parts[0] == SENTINEL and parts[1].isdigit():
                n = int(parts[1])
    except subprocess.TimeoutExpired:
        n = 0
    finally:
        os.unlink(path)
    return n == len(tests)

CTRL_LOG = f'{PHASE5}/control_watch_s{SEED}.jsonl'
_rng = random.Random(SEED)
_call_n = [0]

def random_reward(prompts, completions, test_imports, test_list, **kw):
    rewards = [_rng.uniform(0.0, 1.3) for _ in completions]   # what the trainer sees
    recs = [{'test_imports': ti, 'test_list': tl} for ti, tl in zip(test_imports, test_list)]
    codes = [extract_code(c) for c in completions]
    with ThreadPoolExecutor(max_workers=8) as pool:   # true quality, silent
        flags = list(pool.map(lambda a: run_tests(*a), zip(codes, recs)))
    _call_n[0] += 1
    with open(CTRL_LOG, 'a') as f:
        f.write(json.dumps({'call': _call_n[0], 'true_pass_frac': sum(flags)/len(flags),
                            'mean_random_r': sum(rewards)/len(rewards)}) + '\n')
    return rewards

print('control reward armed: trainer sees Uniform(0,1.3); true pass rate logged to', CTRL_LOG)

In [ ]:
# Train: identical recipe to the real GRPO seed-3407 run, only the reward differs
import torch
from unsloth import FastLanguageModel
try:
    from unsloth import PatchFastRL
    PatchFastRL('GRPO', FastLanguageModel)
except ImportError:
    pass
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
merged = f'/content/merged_s{SEED}'
if not os.path.exists(merged):
    m, t = FastLanguageModel.from_pretrained(
        f'{PHASE3}/sft_notrace_s{SEED}_ep1', max_seq_length=1536,
        load_in_4bit=False, dtype=torch.bfloat16)
    m.save_pretrained_merged(merged, t, save_method='merged_16bit')
    del m, t; gc.collect(); torch.cuda.empty_cache()
model, tok = FastLanguageModel.from_pretrained(
    merged, max_seq_length=1536, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)

def chat(p):
    return tok.apply_chat_template([{'role': 'user', 'content': p}],
                                   tokenize=False, add_generation_prompt=True)

rows = [{'prompt': chat(build_repair_prompt(b['buggy'], b['test_list'])),
         'test_imports': list(b['test_imports']), 'test_list': list(b['test_list'])}
        for b in rl_bugs]
ds = Dataset.from_list(rows).shuffle(seed=SEED)
FastLanguageModel.for_training(model)
cfg = GRPOConfig(
    per_device_train_batch_size=8, gradient_accumulation_steps=4,
    num_generations=8, max_prompt_length=768, max_completion_length=512,
    temperature=1.0, top_p=0.95, beta=0.04,
    learning_rate=5e-6, lr_scheduler_type='cosine', warmup_ratio=0.1,
    max_steps=250, logging_steps=10, seed=SEED, output_dir='/content/out',
    report_to='none', save_strategy='no')
trainer = GRPOTrainer(model=model, processing_class=tok, reward_funcs=random_reward,
                      args=cfg, train_dataset=ds)
trainer.train()
model.save_pretrained(f'{PHASE5}/grpo_random_s{SEED}')
print('control adapter saved. The reward column above is NOISE by design —')
print('what matters is true_pass_frac in', CTRL_LOG, '(flat = model unharmed, falling = noise hurts)')

In [ ]:
# VERDICT: dev eval + the control comparison on the 3407 lineage
def passes_all(code, rec):
    return run_tests(code, rec)

@torch.no_grad()
def sample_k(source_code, test_list, k=16):
    enc = tok([chat(build_repair_prompt(source_code, test_list))],
              return_tensors='pt', padding=True, padding_side='left').to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=k, max_new_tokens=512,
                         pad_token_id=tok.eos_token_id)
    return [extract_code(t) for t in tok.batch_decode(
        out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

FastLanguageModel.for_inference(model)
per_bug = []
for b in dev_bugs:
    codes = sample_k(b['buggy'], b['test_list'])
    with ThreadPoolExecutor(max_workers=8) as pool:
        flags = list(pool.map(lambda c: passes_all(c, b), codes))
    per_bug.append({'id': b['id'], 'n': 16, 'c': sum(flags)})
p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
res = {'tag': f'grpo_random_seed{SEED}', 'pass@1': p1, 'pass@16': p16,
       'gap': p16 - p1, 'per_bug': per_bug}
json.dump(res, open(f'{PHASE5}/dev_eval_grpo_random_seed{SEED}.json', 'w'))

sft = json.load(open(f'{PHASE3}/dev_eval_ep1_seed{SEED}.json'))
real = json.load(open(f'{PHASE5}/dev_eval_grpo_seed{SEED}.json'))
print(f"{'model (3407 lineage)':<24} pass@1   pass@16")
for name, r in (('SFT (baseline)', sft), ('GRPO real reward', real), ('GRPO RANDOM reward', res)):
    print(f"{name:<24} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%")
print('\nReading: RANDOM ~ SFT -> the real gain was caused by the execution signal.')
print('RANDOM ~ real GRPO -> gain was spurious. RANDOM << SFT -> noise damages.')
print('Also check the tail of the control watch (true pass under noise):')
lines = open(CTRL_LOG).read().strip().splitlines()
for ln in lines[:2] + lines[-2:]:
    print(' ', ln)

## Bring back to the session
1. The **three-row control comparison** (SFT / real GRPO / random GRPO)
2. The **first and last control-watch lines** (true pass rate under noise)

After this: notebook 11 — the Phase-7 FINAL EXAM (L4, frozen protocol).